# NER Notebook: RegEx → Gazetteer → ML

In [1]:
import sys, platform
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Executable: c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe
Platform: Windows-10-10.0.26100-SP0


# Step A — Download CoNLL-2003 (manual-friendly)

We’ll try to download `conll2003.zip` from a stable mirror: `https://data.deepai.org/conll2003.zip`.  The automatic download gives too many errors.

- If your network blocks the download **or** you prefer manual:
  1. Create a folder named `data` under your project.
  2. Manually download via browser: https://data.deepai.org/conll2003.zip
  3. Save it as: `data/conll2003.zip`
  
This notebook will **skip the download** if `data/conll2003.zip` already exists.


In [9]:
import os
import urllib.request

data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
zip_path = os.path.join(data_dir, "conll2003.zip")
url = "https://data.deepai.org/conll2003.zip"

if os.path.exists(zip_path):
    print("✅ Found existing:", zip_path)
else:
    try:
        print("Downloading:", url)
        urllib.request.urlretrieve(url, zip_path)
        print("✅ Downloaded to:", zip_path)
    except Exception as e:
        print("⚠️ Download failed.")
        print("Tip: Manually download the file and save it as:", zip_path)
        print("Error:", repr(e))


Downloading: https://data.deepai.org/conll2003.zip
✅ Downloaded to: data\conll2003.zip


# Step B — Extract the ZIP

This will extract the contents into `data/conll2003/`.  
We’ll then list what files are present (some archives use names like `eng.train`, `eng.testa`, `eng.testb`).

In [10]:
import zipfile
import os

extract_dir = os.path.join(data_dir, "conll2003")
os.makedirs(extract_dir, exist_ok=True)

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Missing {zip_path}. If download failed, manually place conll2003.zip under {data_dir}/ and re-run this cell."
    )

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print("✅ Extracted to:", os.path.abspath(extract_dir))

# Walk and show files
found_files = []
for root, _, files in os.walk(extract_dir):
    for f in files:
        found_files.append(os.path.join(root, f))
print("Files found:")
for p in found_files:
    print(" -", p)


✅ Extracted to: c:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003
Files found:
 - data\conll2003\metadata
 - data\conll2003\test.txt
 - data\conll2003\train.txt
 - data\conll2003\valid.txt


# Step C — Locate train/valid/test files

Different releases use different filenames:
- Common: `eng.train`, `eng.testa` (valid), `eng.testb` (test)
- Others: `train.txt`, `valid.txt`, `test.txt`

We’ll search for all known patterns and pick what’s available.


In [11]:
import glob

candidates = {
    "train": ["**/eng.train", "**/train.txt", "**/train"],
    "valid": ["**/eng.testa", "**/valid.txt", "**/validation.txt", "**/dev.txt", "**/valid"],
    "test":  ["**/eng.testb", "**/test.txt", "**/test"]
}

resolved = {}
for split, patterns in candidates.items():
    path = None
    for pat in patterns:
        matches = glob.glob(os.path.join(extract_dir, pat), recursive=True)
        if matches:
            path = matches[0]
            break
    resolved[split] = path

print("Resolved paths:")
for k, v in resolved.items():
    print(f"  {k}: {v}")

# Basic guard
if not resolved["train"] or not resolved["test"]:
    raise RuntimeError("Could not resolve train/test files. Check the printed file list above.")


Resolved paths:
  train: data\conll2003\train.txt
  valid: data\conll2003\valid.txt
  test: data\conll2003\test.txt


# Step D — Parse CoNLL format and preview

CoNLL-2003 (English) has **4 columns** per token (space-separated):
`token  POS  CHUNK  NER`

Sentences are **separated by blank lines**.  
We’ll parse into lists of `(tokens, tags)` per sentence and preview a couple.


In [12]:
def read_conll(path, col_token=0, col_tag=3):
    """Read a CoNLL file (with 4 columns: token, POS, CHUNK, NER)."""
    sentences = []
    tokens, tags = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
                continue
            if line.startswith("-DOCSTART-"):
                # New document marker; treat as sentence boundary
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
                continue
            parts = line.split()
            # robust to lines with unexpected columns
            if len(parts) >= 4:
                tokens.append(parts[col_token])
                tags.append(parts[col_tag])
            else:
                # fallback: assume last is tag, first is token
                tokens.append(parts[0])
                tags.append(parts[-1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_path = resolved["train"]
valid_path = resolved["valid"] or resolved["train"]  # fallback if no explicit valid set
test_path  = resolved["test"]

train_sents = read_conll(train_path)
valid_sents = read_conll(valid_path)
test_sents  = read_conll(test_path)

print(f"Train sentences: {len(train_sents)}")
print(f"Valid sentences: {len(valid_sents)}")
print(f"Test sentences : {len(test_sents)}")

for i in range(2):
    toks, tgs = train_sents[i]
    print(f"\n--- Train example {i} ---")
    print("Tokens:", toks)
    print("Tags:  ", tgs)


Train sentences: 14041
Valid sentences: 3250
Test sentences : 3453

--- Train example 0 ---
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Tags:   ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']

--- Train example 1 ---
Tokens: ['Peter', 'Blackburn']
Tags:   ['B-PER', 'I-PER']


## Display 10 plain training sentences (no tags/hints)
Goal: print 10 random training sentences as raw text; students manually mark likely entities (PER/ORG/LOC/DATE/MISC).  
Prereq: `train_sents` is already loaded from the CoNLL parse step.

## BIO (IOB2) tagging — quick, systematic rules
**Use labels:** `PER, ORG, LOC, MISC, O`  
**Scheme:** BIO (aka IOB2)

1) **Start** every entity span with `B-TYPE`.  
2) **Continue** inside the same span with `I-TYPE`.  
3) **Single-token entities** still use `B-TYPE` (in BIO).  
4) **Outside** any entity → `O`.  
5) **Back-to-back entities:** even if the same type, the next span **restarts** with `B-TYPE`.  
6) **No nesting** (for our exercise): pick the **outermost** or follow your chosen guideline consistently.  
7) **Punctuation** is `O`; include lowercase connectors inside names if they belong (e.g., “Bank **of** England” → all ORG).  
8) **Be consistent** about titles/honorifics (“Dr.”, “Mr.”): decide once (include or exclude) and apply uniformly.

**Examples**  
- “Barack Obama” → `B-PER I-PER`  
- “Bank of England” → `B-ORG I-ORG I-ORG`  
- “Paris” → `B-LOC`  
- “July 14 , 1998” → `B-DATE I-DATE O B-DATE` (comma is `O`)  
- “IBM and Microsoft” → `B-ORG O B-ORG`

**Manual workflow**  
a) Mark spans on plain text first (boundary + type).  
b) Convert spans to tokens: first token `B-TYPE`, remaining `I-TYPE`; everything else `O`.  
c) Recheck transitions: allowed → `O→B`, `B→I (same type)`, `I→I (same type)`, `I→B`, `B→B`.  


In [16]:
# Display 10 plain training sentences (no tags, no hints)

import random, re

def detok(tokens):
    s = " ".join(tokens)
    s = re.sub(r"\s+([.,;:!?%)])", r"\1", s)
    s = re.sub(r"([(])\s+", r"\1", s)
    s = s.replace("`` ", '"').replace(" ''", '"')
    s = s.replace(" 's", "'s").replace(" n't", "n't").replace(" ’s", "’s")
    return s

random.seed(42)
assert 'train_sents' in globals(), "train_sents not found; run the CoNLL parsing cells first."
idxs = random.sample(range(len(train_sents)), k=10)

for i, idx in enumerate(idxs, 1):
    tokens, _ = train_sents[idx]
    print(f"{i}. {detok(tokens)}")


1. CINCINNATI AT COLORADO
2. Relations with Russia, which is our main partner, have great importance, " Kuchma said.
3. He added: " If no one asked, I never opened my mouth.
4. -- Steve Weizman, Copenhagen newsroom +45 33969650
5. 70 71 74
6. Graafschap Doetinchem 3 RKC Waalwijk 2
7. Nottingham Forest 1 Middlesbrough 1
8. 69 66, Jose Maria Canizares (Spain) 67 68, Paul Lawrie
9. Hwa Kay plunges on rights issue, earnings.
10. But it was the bullish profit forecast for 1996/97 that took the spotlight in the market, with some analysts saying 20 percent may even be an understatement.


## Heuristic patterns (for manual NER) and their target regexes
Goal: Give system concrete clues and ask to find all patterns and corresponding RegEx

- **CAPITALIZED_TOKEN (proper-noun-ish single word):** word with initial capital even mid-sentence.  
    **ALL_CAPS_TOKEN:** all-caps tokens (teams, tickers, acronyms).  
- **MULTIWORD_CAPITALIZED (2–4 tokens):** likely PER/ORG/LOC.  
- **PROPER_WITH_OF/AND:** capitalized spans with connectors “of/and/&” (e.g., *Bank of England*).  

## Pattern mining: print **Pattern | Example | Regex** from training
We scan `train_sents`, detokenize each sentence, and for each heuristic pattern (proper nouns, all-caps, multiword names, org-with-suffix, years, year ranges, percents, phone numbers, simple scores), we print the **first example** found alongside its **regex**. If nothing is found, we print “—”.

In [17]:
import re

# Guard: expect train_sents from prior CoNLL parsing
assert 'train_sents' in globals(), "train_sents not found; run the CoNLL parsing cells first."

def detok(tokens):
    s = " ".join(tokens)
    s = re.sub(r"\s+([.,;:!?%)])", r"\1", s)
    s = re.sub(r"([(])\s+", r"\1", s)
    s = s.replace("`` ", '"').replace(" ''", '"')
    s = s.replace(" 's", "'s").replace(" n't", "n't").replace(" ’s", "’s")
    return s

# Heuristic regex patterns (text-level; adjust as needed)
patterns = {
    "CAPITALIZED_TOKEN": r"\b[A-Z][a-z]+(?:-[A-Z][a-z]+)?\b",
    "ALL_CAPS_TOKEN": r"\b[A-Z]{2,}\b",
    "MULTIWORD_CAPITALIZED": r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\b",
    "PROPER_WITH_OF/AND/&": r"\b[A-Z][\w.-]*(?:\s+(?:of|and|&)\s+[A-Z][\w.-]*)+\b",
    "ORG_SUFFIX": r"\b(?:[A-Z][\w.&-]*\s+){0,3}(?:Bank|Ltd|Inc|University|Committee|Council|Party)\b",
    "PERSON_LIKE": r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2}\b",
    "HONORIFIC_PERSON": r"\b(?:Mr|Ms|Mrs|Dr|Prof|Sir|Lady)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b",
    "YEAR": r"\b(?:19|20)\d{2}\b",
    "YEAR_RANGE": r"\b(?:19|20)\d{2}[/\-](?:\d{2}|\d{4})\b",
    "PERCENT": r"\b\d{1,3}(?:\.\d+)?\s?(?:%|percent)\b",
    "PHONE_INTL": r"\+\d{2,3}\s\d[\d\s-]{5,}",
    "SCORE_SIMPLE": r"\b\d+\s*[-–]\s*\d+\b",
}

compiled = {name: re.compile(rx) for name, rx in patterns.items()}

def first_match_example(rx: re.Pattern, sentences) -> str | None:
    for tokens, _ in sentences:
        text = detok(tokens)
        m = rx.search(text)
        if m:
            return m.group(0)
    return None

rows = []
for name, rx in compiled.items():
    ex = first_match_example(rx, train_sents) or "—"
    rows.append((name, ex, rx.pattern))

# Pretty print (fixed width columns)
col1, col2 = 24, 40
print(f"{'PATTERN':{col1}} | {'EXAMPLE':{col2}} | REGEX")
print("-" * (col1 + col2 + 10 + 6))
for name, ex, rx in rows:
    ex_short = (ex[:col2-3] + "...") if len(ex) > col2 else ex
    print(f"{name:{col1}} | {ex_short:{col2}} | {rx}")


PATTERN                  | EXAMPLE                                  | REGEX
--------------------------------------------------------------------------------
CAPITALIZED_TOKEN        | German                                   | \b[A-Z][a-z]+(?:-[A-Z][a-z]+)?\b
ALL_CAPS_TOKEN           | EU                                       | \b[A-Z]{2,}\b
MULTIWORD_CAPITALIZED    | Peter Blackburn                          | \b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\b
PROPER_WITH_OF/AND/&     | Britain and France                       | \b[A-Z][\w.-]*(?:\s+(?:of|and|&)\s+[A-Z][\w.-]*)+\b
ORG_SUFFIX               | Kurdistan Democratic Party               | \b(?:[A-Z][\w.&-]*\s+){0,3}(?:Bank|Ltd|Inc|University|Committee|Council|Party)\b
PERSON_LIKE              | Peter Blackburn                          | \b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2}\b
HONORIFIC_PERSON         | Sir Mark Prescott                        | \b(?:Mr|Ms|Mrs|Dr|Prof|Sir|Lady)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b
YEAR            

## Build a gazetteer from training data (PER/ORG/LOC/MISC) and print samples
We extract gold entity spans from `train_sents`, count unique full-surface forms per type, and print top examples (also save TSVs under `data/gazetteer/`).

In [18]:
import os, re
from collections import Counter, defaultdict

# Guard
assert 'train_sents' in globals(), "train_sents not found; run the CoNLL parsing cells first."

def iob_to_spans(tokens, tags):
    """Return list of (start, end, type) spans from IOB2 tags."""
    spans = []
    curr_type, start = None, None
    for i, tag in enumerate(tags):
        if tag == "O" or not tag:
            if curr_type is not None:
                spans.append((start, i, curr_type))
                curr_type, start = None, None
            continue
        # split tag
        if "-" in tag:
            prefix, etype = tag.split("-", 1)
        else:
            prefix, etype = "I", tag
        if prefix == "B" or (curr_type is not None and etype != curr_type):
            if curr_type is not None:
                spans.append((start, i, curr_type))
            curr_type, start = etype, i
        else:  # I-etype
            if curr_type is None:
                curr_type, start = etype, i
    if curr_type is not None:
        spans.append((start, len(tags), curr_type))
    return spans

def detok(tokens):
    s = " ".join(tokens)
    s = re.sub(r"\s+([.,;:!?%)])", r"\1", s)
    s = re.sub(r"([(])\s+", r"\1", s)
    return s

# Build gazetteers
gaz = defaultdict(Counter)  # type -> Counter(text -> freq)

for tokens, tags in train_sents:
    for s, e, etype in iob_to_spans(tokens, tags):
        if etype not in ("PER","ORG","LOC","MISC"):
            continue
        text = detok(tokens[s:e]).strip()
        if text:  # simple sanity filter
            gaz[etype][text] += 1

# Save to disk
out_dir = os.path.join("data", "gazetteer")
os.makedirs(out_dir, exist_ok=True)
for etype in ("PER","ORG","LOC","MISC"):
    path = os.path.join(out_dir, f"{etype}.tsv")
    with open(path, "w", encoding="utf-8") as f:
        for name, cnt in gaz[etype].most_common():
            f.write(f"{name}\t{cnt}\n")

# Print summary + samples
def print_samples(etype, k=15):
    print(f"\n=== {etype} ===  unique={len(gaz[etype])}")
    for name, cnt in gaz[etype].most_common(k):
        print(f"- {name}  ({cnt})")

print(f"Saved gazetteers to: {os.path.abspath(out_dir)}")
for t in ("PER","ORG","LOC","MISC"):
    print_samples(t, k=15)


Saved gazetteers to: c:\Users\murth\Desktop\nlpSession02\codeBase\data\gazetteer

=== PER ===  unique=3614
- Clinton  (70)
- Dole  (36)
- Arafat  (35)
- Yeltsin  (32)
- Lebed  (30)
- Dutroux  (30)
- Wasim Akram  (29)
- Waqar Younis  (25)
- Mushtaq Ahmed  (25)
- Mother Teresa  (20)
- Croft  (20)
- Aamir Sohail  (19)
- Bill Clinton  (19)
- Mullally  (18)
- Netanyahu  (16)

=== ORG ===  unique=2403
- Reuters  (73)
- U.N.  (35)
- NEW YORK  (30)
- CHICAGO  (30)
- PUK  (25)
- OSCE  (25)
- EU  (23)
- NATO  (22)
- European Union  (20)
- Honda  (20)
- Interfax  (18)
- Ajax  (18)
- KDP  (17)
- BALTIMORE  (17)
- MINNESOTA  (17)

=== LOC ===  unique=1332
- U.S.  (303)
- Germany  (141)
- Britain  (133)
- Australia  (130)
- England  (123)
- France  (122)
- Spain  (110)
- Italy  (98)
- LONDON  (93)
- China  (91)
- Russia  (88)
- Japan  (87)
- Pakistan  (85)
- Sweden  (81)
- Belgium  (71)

=== MISC ===  unique=871
- Russian  (92)
- German  (80)
- British  (73)
- French  (59)
- Dutch  (50)
- GMT  (49)


## BIO (aka IOB2) and a “BIO fixer” — what & why

**BIO/IOB2 scheme**  
- **B–TYPE** = *Begin* a span of entity TYPE (e.g., `B-PER`).  
- **I–TYPE** = *Inside* the same span (continuation).  
- **O** = *Outside* any entity.  
- **Single-token entities** still use **B–TYPE** (not I–TYPE).  
- **Legal transitions (per token):**  
  - `O → B-X` (start a span)  
  - `B-X → I-X` (continue) or `B-X → B-Y` (new span)  
  - `I-X → I-X` (continue) or `I-X → B-Y` (new span) or `I-X → O` (end)

**Problem without sequence models:** token classifiers (RF/SVM/MaxEnt) predict labels independently, so you often get **illegal** or **messy** sequences, e.g. `O I-PER` (illegal start), `I-ORG` after `I-PER`, or broken spans around punctuation.

**BIO fixer (post-processor)**  
A small rule-based pass that **repairs** predictions to obey BIO and improve spans. Typical rules:
1. **Illegal I → B:** if previous tag is `O` or previous type differs (`I-ORG` after `O` or `I-PER`) → flip to `B-ORG`.  
2. **Type switch inside span:** `I-X` followed by `I-Y` (X≠Y) → start new span: make the second `B-Y`.  
3. **Boundary cleanup:** ensure spans don’t swallow stray punctuation; break at obvious separators unless configured (e.g., keep hyphen inside `ACME-Corp`).  
4. **Merge heuristics:** join adjacent same-type tokens split by minor punctuation/connector you allow (configurable: hyphen, ampersand, “of/and/&” for ORG).  
5. **Gazetteer overrides (optional):** if a longest multiword gazetteer match conflicts with token labels, prefer the gazetteer and set `B/I` accordingly.  
6. **Length sanity (optional):** drop absurd 1-char entities, or cap span length unless whitelisted.

**Before → After example**  
Tokens: `["Bank", "of", "England", "said", "."]`  
Raw preds: `["B-ORG","O","I-ORG","O","."]` (splits “of” out of the org)  
Fixer: allow connector “of” inside ORG →  
Final: `["B-ORG","I-ORG","I-ORG","O","O"]`  → one clean ORG span.

> Outcome: you get **legal BIO sequences** and **cleaner entity boundaries**, which meaningfully boosts span-level F1.

Say **“next”** if you want me to add a small BIO-fixer pass + span-F1 evaluator to your notebook.


## What is an NP? What do we do with it for NER? (with example)
**NP (Noun Phrase)** = a phrase whose head is a noun/proper noun, e.g., *“Barack Obama”*, *“the central bank”*, *“Bank of England”*, *“London”*.  
In classic (“pre-HMM/CRF”) NER, a strong approach was **NP-first NER**: find NPs with a shallow parser, then **classify each NP span** as `PER/ORG/LOC/MISC/O`, and finally project that back to BIO tags.

### Why NP-first?
- Entities usually align with NPs; classifying spans avoids per-token noise.
- You can use rich **span features** (head word, suffixes, shape, gazetteer hits, context words) and then assign one label per NP.

### What we’d do
1) **POS tag + NP chunk** the sentence (shallow parsing).  
2) **Generate NP candidates** (often “base NPs”: non-recursive chunks).  
3) **Classify each NP** → `PER/ORG/LOC/MISC/O` using features (orthography, suffixes like *Bank/Ltd/Inc*, gazetteer matches, left/right context words, length).  
4) **Project to BIO**: NP labeled `ORG` becomes `B-ORG I-ORG …` across its tokens.  
5) **Fix boundaries** with small rules (e.g., allow connectors like “of/&/de” inside org names when appropriate).

### Example
Sentence: *“President Barack Obama met executives at Bank of England in London.”*  
POS (simplified): `President/NNP Barack/NNP Obama/NNP met/VBD executives/NNS at/IN Bank/NNP of/IN England/NNP in/IN London/NNP ./.`  
NP chunks (illustrative):
- `[NP President Barack Obama]` → **PER**  
- `[NP executives]` → **O**  
- `[NP Bank of England]` → **ORG**  *(we allow “of” to stay inside)*  
- `[NP London]` → **LOC**

BIO projection:
- `President/B-PER Barack/I-PER Obama/I-PER met/O executives/O at/O Bank/B-ORG of/I-ORG England/I-ORG in/O London/B-LOC ./O`

**Bottom line:** NP-first NER = *chunk NPs → classify spans → convert to BIO*. It was a practical, accurate pipeline before sequence models took over.


# Plan: POS + Regex + Gazetteer + NP Chunk + BIO-fix → Multinomial Logistic Regression (token-level)

## Objective
Build a faithful pre-CRF pipeline: hand-engineered features + fast classifier (MaxEnt/LogReg), then a light **BIO fixer**. Report span-level F1 for true NER quality.

## Feature Set (grouped)
**A. Orthography & Morphology**
- `lower`, `isupper`, `istitle`, `isdigit`
- **word shape** (e.g., `Xxxx`, `XX-XX`, `X.x.`, `d, dd, dddd`)
- Prefixes & suffixes (1–4 chars), `has_hyphen`, `has_dot`, `has_ampersand`, `has_apostrophe`
- Token length buckets (1,2,3–5,6–10,>10)

**B. Context (±1, optionally ±2)**
- Prev/next token: `lower`, `isupper`, `istitle`, `shape`
- BOS/EOS flags
- Bigram shape cues: current+next both Titlecase (`bi_title`), prev+current both ALLCAPS, etc.

**C. Regex Cues (token-level)**
- `CAPITALIZED_TOKEN` (`^[A-Z][a-z]+(?:-[A-Z][a-z]+)?$`)
- `ALL_CAPS` (`^[A-Z]{2,}$`)
- `YEAR` (`^(?:19|20)\d{2}$`)
- `PERCENT` (`^\d{1,3}(?:\.\d+)?%$`)
- Optional: `HAS_DIGITS`, `HAS_MIXED_ALNUM`, currency symbol, email/URL/phone formats (rare but useful)

**D. Gazetteer Features**
- **Unigram gaz hits**: membership in `PER/ORG/LOC/MISC` single-token lists (from train + optional external lists)
- **Longest multiword gaz hit** *projected to token level*: for each token, length/type of the longest gaz match covering it (e.g., `gaz_len=3`, `gaz_type=ORG`)
- **Suffix priors** (from gaz statistics): last token in span equals common ORG/LOC suffix (`Bank`, `Ltd`, `Inc`, `University`, `City`, etc.)

**E. POS & NP-Chunk Features (shallow parsing)**
- POS tags: current/prev/next POS (e.g., `NNP`, `NNS`, `IN`, `DT`)
- NP-chunk tags: `B-NP/I-NP/O` (IOB for base noun phrases)
- NP context: inside an NP?, NP length (1–4,>4), NP head/last token POS/shape, NP begins with determiner?, NP contains connector (`of/and/&/de`)?

**F. Sentence/Doc Cues (optional)**
- Sentence position bucket (0–10%, …, 90–100%)
- All-caps headline heuristic (newswire), if detectable

## Model & Training
- **Multinomial Logistic Regression** (`solver='saga'`, `penalty='l2'`, `max_iter≈1000`, `n_jobs=-1`)
- `DictVectorizer` over the above features
- Optionally tune `C` via quick CV on a held-out subset
- Class weights (balanced) if label skew hurts minority classes

## Decoding: BIO Fixer (post-processing on predictions)
- Enforce legal BIO: `O→I` ⇒ flip to `B`; type switches inside span ⇒ start new `B`
- Merge same-type tokens across allowed connectors (`-`, `&`, `of`, `de`) when confident
- Keep punctuation out unless whitelisted
- **Gazetteer override (optional)**: if a strong multiword gaz hit conflicts, rewrite the local BIO to match the gaz span/type

## Evaluation
- **Primary (as you asked): token-level micro-F1** across `O` + entity labels
- **Recommended add-on:** span-level micro-F1 (exact boundary+type), plus per-type F1 (PER/ORG/LOC/MISC)
- Sanity: confusion matrix over labels, precision/recall by type

## Deliverables (when we implement)
1) POS/Chunk tagger setup inside venv (lightweight)  
2) Feature extractor (as above), vectorizer, LogReg training  
3) BIO-fixer pass  
4) Metrics: span-F1 printed neatly

## Risks & Mitigations
- **Speed**: POS/chunk adds cost → use a lightweight tagger; cache outputs  
- **Noisy gazetteers**: restrict overrides to high-confidence (longer spans, frequent entries)  
- **Class imbalance**: use `class_weight='balanced'` or tune `C`  

**If this looks right, I’ll produce the next block with the exact Markdown+Code to implement it, step-by-step.**


In [21]:
# Self-contained: POS+NP features → Logistic Regression → BIO-fix (no metrics)

import sys, subprocess, re, os, time
from collections import Counter, defaultdict

# --- deps ---
try:
    import nltk
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk

try:
    from sklearn.feature_extraction import DictVectorizer
    from sklearn.linear_model import LogisticRegression
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.feature_extraction import DictVectorizer
    from sklearn.linear_model import LogisticRegression

assert 'train_sents' in globals() and 'valid_sents' in globals(), "Run the CoNLL parsing cells first."

# --- NLTK tagger & chunker ---
# Try new then old resource names
for res in ("averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"):
    try:
        nltk.data.find(f"taggers/{res}")
        POS_RES = res
        break
    except LookupError:
        try:
            nltk.download(res, quiet=True)
            POS_RES = res
            break
        except Exception:
            POS_RES = None
if POS_RES is None:
    nltk.download("averaged_perceptron_tagger", quiet=True)

from nltk import pos_tag
from nltk.chunk import RegexpParser

_CHUNK_GRAMMAR = r"""
  NP: {<DT|PRP\$>?<JJ.*>*<NN.*|NNP.*>+}
"""
_CHUNKER = RegexpParser(_CHUNK_GRAMMAR)

def pos_tag_sentence(tokens):
    return pos_tag(tokens)

def np_iob_tags(pos_tags):
    """Return B-NP/I-NP/O aligned to pos_tags using a simple base-NP grammar."""
    tree = _CHUNKER.parse(pos_tags)
    iob = []
    for node in tree:
        if isinstance(node, tuple):
            iob.append("O")
        else:
            if node.label() == "NP":
                for j, _ in enumerate(node):
                    iob.append("B-NP" if j == 0 else "I-NP")
            else:
                for _ in node:
                    iob.append("O")
    return iob

# --- Gazetteer (reuse if exists, else build from train) ---
def iob_to_spans(tokens, tags):
    spans=[]; curr=None; start=None
    for i, tag in enumerate(tags):
        if tag == "O" or not tag:
            if curr is not None:
                spans.append((start, i, curr)); curr=None; start=None
            continue
        pref, typ = (tag.split("-",1) + [""])[:2] if "-" in tag else ("I", tag)
        if pref == "B" or (curr is not None and typ != curr):
            if curr is not None: spans.append((start, i, curr))
            curr, start = typ, i
        else:
            if curr is None: curr, start = typ, i
    if curr is not None: spans.append((start, len(tags), curr))
    return spans

if 'gaz' not in globals():
    gaz = defaultdict(Counter)
    for toks, tgs in train_sents:
        for s,e,typ in iob_to_spans(toks,tgs):
            if typ in ("PER","ORG","LOC","MISC"):
                gaz[typ][" ".join(toks[s:e])] += 1

gaz_unigram = {k: {name for name in gaz[k] if len(name.split())==1} for k in ("PER","ORG","LOC","MISC")}
gaz_multi   = {k: {name for name in gaz[k] if len(name.split())>=2} for k in ("PER","ORG","LOC","MISC")}
MAX_GAZ_LEN = 5

# --- Regex cues ---
RX = {
    "CAPITALIZED_TOKEN": re.compile(r"^[A-Z][a-z]+(?:-[A-Z][a-z]+)?$"),
    "ALL_CAPS":          re.compile(r"^[A-Z]{2,}$"),
    "YEAR":              re.compile(r"^(?:19|20)\d{2}$"),
    "PERCENT":           re.compile(r"^\d{1,3}(?:\.\d+)?%$"),
}

def is_title(w): return bool(w[:1].isupper() and (len(w)==1 or w[1:].islower()))
def word_shape(w: str) -> str:
    s = ''.join('X' if c.isupper() else 'x' if c.islower() else 'd' if c.isdigit() else c for c in w)
    out=[]; 
    for ch in s:
        if not out or out[-1]!=ch: out.append(ch)
    return ''.join(out)

def gaz_coverage(tokens):
    n = len(tokens)
    cov_type = [None]*n
    cov_len  = [0]*n
    for i in range(n):
        for L in range(min(MAX_GAZ_LEN, n-i), 2-1, -1):  # L from max..2
            span = " ".join(tokens[i:i+L])
            hit = None
            for typ in ("PER","ORG","LOC","MISC"):
                if span in gaz_multi[typ]:
                    hit = typ; break
            if hit:
                for k in range(i, i+L):
                    if L > cov_len[k]:
                        cov_len[k] = L
                        cov_type[k] = hit
    return cov_type, cov_len

def bio_fix(pred_tags, tokens):
    fixed = []
    prev_type = None
    for tag in pred_tags:
        if tag == "O" or not tag:
            fixed.append("O"); prev_type = None; continue
        pref, typ = (tag.split("-",1) + [""])[:2] if "-" in tag else ("I", tag)
        if pref == "I" and prev_type is None:
            pref = "B"
        elif pref == "I" and typ != prev_type:
            pref = "B"
        fixed.append(f"{pref}-{typ}")
        prev_type = typ
    return fixed

def token_features(sent_tokens, pos_tags, np_iob, cov_type, cov_len, i):
    w = sent_tokens[i]
    prev = sent_tokens[i-1] if i>0 else "<BOS>"
    nextw= sent_tokens[i+1] if i+1<len(sent_tokens) else "<EOS>"
    pos  = pos_tags[i][1]
    pos_p= pos_tags[i-1][1] if i>0 else "<BOS>"
    pos_n= pos_tags[i+1][1] if i+1<len(sent_tokens) else "<EOS>"
    np_t = np_iob[i]
    return {
        "w.lower": w.lower(),
        "w.isupper": w.isupper(),
        "w.istitle": is_title(w),
        "w.isdigit": w.isdigit(),
        "w.shape": word_shape(w),
        "pref1": w[:1], "pref2": w[:2], "pref3": w[:3], "pref4": w[:4],
        "suf1": w[-1:], "suf2": w[-2:], "suf3": w[-3:], "suf4": w[-4:],
        "len.bucket": 1 if len(w)<=1 else 2 if len(w)==2 else 3 if len(w)<=5 else 4 if len(w)<=10 else 5,
        "has_hyphen": "-" in w, "has_dot": "." in w, "has_amp": "&" in w, "has_apos": "'" in w,
        "prev.lower": prev.lower(), "prev.istitle": is_title(prev), "prev.isupper": prev.isupper(),
        "next.lower": nextw.lower(), "next.istitle": is_title(nextw), "next.isupper": nextw.isupper(),
        "BOS": i==0, "EOS": i==len(sent_tokens)-1, "bi_title": is_title(w) and is_title(nextw),
        "pos": pos, "pos_prev": pos_p, "pos_next": pos_n, "np_iob": np_t,
        "rx_capitalized": bool(RX["CAPITALIZED_TOKEN"].match(w)),
        "rx_allcaps": bool(RX["ALL_CAPS"].match(w)),
        "rx_year": bool(RX["YEAR"].match(w)),
        "rx_percent": bool(RX["PERCENT"].match(w)),
        "gaz_PER1": w in gaz_unigram["PER"],
        "gaz_ORG1": w in gaz_unigram["ORG"],
        "gaz_LOC1": w in gaz_unigram["LOC"],
        "gaz_MISC1": w in gaz_unigram["MISC"],
        "gaz_cover": cov_type[i] or "NONE",
        "gaz_len": cov_len[i],
    }

def build_xy(sents):
    X, y, lengths = [], [], []
    for tokens, tags in sents:
        ptags = pos_tag_sentence(tokens)
        npiob = np_iob_tags(ptags)
        cov_t, cov_l = gaz_coverage(tokens)
        for i in range(len(tokens)):
            X.append(token_features(tokens, ptags, npiob, cov_t, cov_l, i))
            y.append(tags[i])
        lengths.append(len(tokens))
    return X, y, lengths

# --- Train & predict ---
t0 = time.time()
X_train, y_train, _         = build_xy(train_sents)
X_valid, y_valid, val_lens  = build_xy(valid_sents)

vec = DictVectorizer(sparse=True)
Xtr = vec.fit_transform(X_train)
Xva = vec.transform(X_valid)

clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    multi_class="multinomial",
    max_iter=1000
)
clf.fit(Xtr, y_train)
train_time = time.time() - t0

y_pred = clf.predict(Xva)

# BIO-fix per sentence
pred_fixed = []
off = 0
for L, (tokens, _) in zip(val_lens, valid_sents):
    seg = y_pred[off:off+L]
    pred_fixed.extend(bio_fix(seg.tolist(), tokens))
    off += L

print(f"✅ Trained Logistic Regression in {train_time:.2f}s")
print(f"   Train tokens: {len(y_train):,} | Valid tokens: {len(y_valid):,}")
print(f"   Feature dimension: {Xtr.shape[1]:,} | Classes: {sorted(set(y_train))}")
print("   Predictions computed and BIO-fixed (metrics intentionally not printed).")


c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✅ Trained Logistic Regression in 220.27s
   Train tokens: 203,621 | Valid tokens: 51,362
   Feature dimension: 101,581 | Classes: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
   Predictions computed and BIO-fixed (metrics intentionally not printed).


## Where to put this (brief): add **one new cell below** your previous training cell
- **Do not edit** the earlier cell.  
- Insert this **new cell right after** the “Trained Logistic Regression…” printout.  
- Run it once: it re-fits the model **without `multi_class`** and re-applies the BIO-fix, silencing the warning.  
- (Optional long-term: in your earlier cell, just remove `multi_class='multinomial'` from the `LogisticRegression(...)` line, then you won’t need this patch cell.)


In [22]:
# Patch cell: retrain without the deprecated 'multi_class' arg and re-apply BIO-fix

import time
from sklearn.linear_model import LogisticRegression

# Reuse existing variables from the previous cell
assert all(v in globals() for v in ["Xtr","y_train","Xva","valid_sents","bio_fix"]), "Run the previous training cell first."

clf = LogisticRegression(solver="saga", penalty="l2", max_iter=1000)

t0 = time.time()
clf.fit(Xtr, y_train)
y_pred = clf.predict(Xva)

# BIO-fix per sentence
pred_fixed, off = [], 0
for tokens, _ in valid_sents:
    L = len(tokens)
    seg = y_pred[off:off+L]
    pred_fixed.extend(bio_fix(seg.tolist(), tokens))
    off += L

print(f"✅ Re-trained without 'multi_class' in {time.time()-t0:.2f}s (warning silenced).")


✅ Re-trained without 'multi_class' in 173.90s (warning silenced).


## Report **span-level** NER quality (micro P/R/F1 + per-type) after BIO-fix
This computes **entity-level** precision/recall/F1 with **exact span + type** match (gold from `valid_sents`, preds from `pred_fixed`). It also prints per-type F1 (PER/ORG/LOC/MISC) and span counts.  


In [23]:
from collections import defaultdict

# Ensure predictions exist; rebuild if needed
if "pred_fixed" not in globals():
    assert all(v in globals() for v in ["clf","Xva","valid_sents","bio_fix"]), "Run training + BIO-fix first."
    y_pred = clf.predict(Xva)
    pred_fixed, off = [], 0
    for tokens, _ in valid_sents:
        L = len(tokens)
        seg = y_pred[off:off+L]
        pred_fixed.extend(bio_fix(seg.tolist(), tokens))
        off += L

def bio_to_spans(tags):
    """Convert BIO tags to spans (start, end_exclusive, type)."""
    spans = []
    cur_type, start = None, None
    for i, tag in enumerate(tags):
        if tag == "O" or not tag:
            if cur_type is not None:
                spans.append((start, i, cur_type))
                cur_type, start = None, None
            continue
        if "-" in tag:
            pref, typ = tag.split("-", 1)
        else:
            pref, typ = "I", tag
        if pref == "B" or (cur_type is not None and typ != cur_type):
            if cur_type is not None:
                spans.append((start, i, cur_type))
            cur_type, start = typ, i
        else:
            if cur_type is None:
                cur_type, start = typ, i
    if cur_type is not None:
        spans.append((start, len(tags), cur_type))
    return spans

# Micro counts
tp = fp = fn = 0
tp_by = defaultdict(int); fp_by = defaultdict(int); fn_by = defaultdict(int)

off = 0
lens = [len(toks) for toks, _ in valid_sents]
for s_idx, L in enumerate(lens):
    gold_tags = valid_sents[s_idx][1]
    pred_tags = pred_fixed[off:off+L]
    off += L

    g = set(bio_to_spans(gold_tags))
    p = set(bio_to_spans(pred_tags))

    tp_set = p & g
    fp_set = p - g
    fn_set = g - p

    tp += len(tp_set); fp += len(fp_set); fn += len(fn_set)
    for _, _, t in tp_set: tp_by[t] += 1
    for _, _, t in fp_set: fp_by[t] += 1
    for _, _, t in fn_set: fn_by[t] += 1

# Micro metrics
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = (2*prec*rec)/(prec+rec) if (prec+rec) else 0.0

print("=== Span-level (exact) micro metrics ===")
print(f"Gold spans: {tp+fn} | Pred spans: {tp+fp} | TP: {tp}  FP: {fp}  FN: {fn}")
print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")

# Per-type
types = sorted(set(list(tp_by.keys()) + list(fp_by.keys()) + list(fn_by.keys())))
if types:
    print("\n=== Per-type span metrics ===")
    for t in types:
        tp_t, fp_t, fn_t = tp_by[t], fp_by[t], fn_by[t]
        p_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else 0.0
        r_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else 0.0
        f_t = (2*p_t*r_t)/(p_t+r_t) if (p_t+r_t) else 0.0
        support = tp_t + fn_t
        print(f"{t:>5}: P={p_t:.4f} R={r_t:.4f} F1={f_t:.4f}  (support={support})")


=== Span-level (exact) micro metrics ===
Gold spans: 5942 | Pred spans: 4676 | TP: 3961  FP: 715  FN: 1981
Precision: 0.8471  Recall: 0.6666  F1: 0.7461

=== Per-type span metrics ===
  LOC: P=0.9205 R=0.8514 F1=0.8846  (support=1837)
 MISC: P=0.9094 R=0.7950 F1=0.8484  (support=922)
  ORG: P=0.7250 R=0.6644 F1=0.6934  (support=1341)
  PER: P=0.8206 R=0.4197 F1=0.5553  (support=1842)
